# Week 6: Experiments and Statistics

In this practical you will conduct experiments to tune the parameters of an EA, learn how to present the results and conduct some statistical testing to determine whether an operator has a signficant effect on the results.

In this tutorial you will learn:

1. How to design and conduct experiments to collect data to investigate the influence of parameters
2. How to calculate statistics from the data
3. How to plot the data using boxplots
4. How to conduct statistical testing using an appropriate test

We will use the EASimple to solve the One-Max problem as a test bed.

In [ ]:
# Install dependencies if needed
!pip install pandas scipy deap numpy matplotlib

## Set up Framework

Let's define the base EA configuration.

In [ ]:
import random
import pandas as pd
import scipy
import numpy as np
import matplotlib.pyplot as plt
from deap import base, creator, tools, algorithms
from scipy.stats import mannwhitneyu, ttest_ind, shapiro

# Safeguard creator to avoid runtime re-definition errors
if not hasattr(creator, "FitnessMax"):
    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
if not hasattr(creator, "Individual"):
    creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, 100)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def evalOneMax(individual):
    return sum(individual),

toolbox.register("evaluate", evalOneMax)
toolbox.register("mate", tools.cxOnePoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.1)
toolbox.register("select", tools.selTournament, tournsize=2)

# Global baseline hyperparameters
MUTPB = 0.1
CXPB = 0.6
POPSIZE = 50
NGEN = 200
TNSIZE = 2

In [ ]:
def runEA(custom_popsize=None, custom_mutpb=None, custom_cxpb=None):
    p_size = custom_popsize if custom_popsize is not None else POPSIZE
    m_prob = custom_mutpb if custom_mutpb is not None else MUTPB
    c_prob = custom_cxpb if custom_cxpb is not None else CXPB
    
    pop = toolbox.population(n=p_size)
    hof = tools.HallOfFame(1)
    
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("std", np.std)
    stats.register("min", np.min)
    stats.register("max", np.max)
    
    pop, log = algorithms.eaSimple(pop, toolbox, cxpb=c_prob, mutpb=m_prob, ngen=NGEN, 
                                   stats=stats, halloffame=hof, verbose=False)
    return pop, log, hof

## Baseline Experiment: Population Size Analysis

In [ ]:
all_logs = []
bests_list = []

for POPSIZE_TEST in [50, 100, 150]:
    for reps in range(10):
        pop, log, hof = runEA(custom_popsize=POPSIZE_TEST)
        best = hof[0].fitness.values[0]
        max_history = log.select('max')
        
        for gen in range(NGEN):
            if max_history[gen] == best:
                break
        
        bests_list.append({'popsize': POPSIZE_TEST, 'fitness': best, 'genMaxFound': gen})
        all_logs.append(log)

df_Bests = pd.DataFrame(bests_list)
print(df_Bests.groupby('popsize').mean())

## Task 1: Exploring the effect of mutation rate

Fixing popsize=100, cxpb=0.6, testing mutation rates from 0.0 to 1.0 in steps of 0.2.

In [ ]:
mutation_results = []
mutation_rates = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

for mut in mutation_rates:
    toolbox.register("mutate", tools.mutFlipBit, indpb=mut)
    for reps in range(10):
        pop, log, hof = runEA(custom_popsize=100, custom_mutpb=mut)
        best = hof[0].fitness.values[0]
        mutation_results.append({'mutation_rate': mut, 'fitness': best})

df_mutation = pd.DataFrame(mutation_results)
df_mutation.boxplot(column=['fitness'], by=['mutation_rate'])
plt.title('Mutation Rate Performance')
plt.show()

In [ ]:
# Statistical Significance Test between best and worst mutation rates
summary = df_mutation.groupby('mutation_rate').mean()
best_mut = summary['fitness'].idxmax()
worst_mut = summary['fitness'].idxmin()

data_best = df_mutation.fitness[df_mutation.mutation_rate == best_mut].tolist()
data_worst = df_mutation.fitness[df_mutation.mutation_rate == worst_mut].tolist()

_, p_best = shapiro(data_best)
_, p_worst = shapiro(data_worst)

print(f"Shapiro Normality p-values -> Best ({best_mut}): {p_best:.4f}, Worst ({worst_mut}): {p_worst:.4f}")

if p_best > 0.05 and p_worst > 0.05:
    _, p_test = ttest_ind(data_best, data_worst)
    print(f"T-Test Evaluation p-value: {p_test:.6f}")
else:
    _, p_test = mannwhitneyu(data_best, data_worst)
    print(f"Mann-Whitney Test Evaluation p-value: {p_test:.6f}")

if p_test < 0.05:
    print("Result: Reject Null Hypothesis. Significant differences observed.")
else:
    print("Result: Cannot Reject Null Hypothesis. No significant performance gap.")

# Task 2: Exploring crossover types:

Evaluating 1-point, 2-point, and Uniform crossover mechanics.

In [ ]:
crossover_results = []
cx_operators = {
    '1-Point': tools.cxOnePoint,
    '2-Point': tools.cxTwoPoint,
    'Uniform': lambda ind1, ind2: tools.cxUniform(ind1, ind2, indpb=0.5)
}

# Restore default baseline mutation to framework state
toolbox.register("mutate", tools.mutFlipBit, indpb=0.1)

for name, op in cx_operators.items():
    toolbox.register("mate", op)
    for reps in range(10):
        pop, log, hof = runEA(custom_popsize=100)
        best = hof[0].fitness.values[0]
        crossover_results.append({'crossover_type': name, 'fitness': best})

df_crossover = pd.DataFrame(crossover_results)
df_crossover.boxplot(column=['fitness'], by=['crossover_type'])
plt.title('Crossover Operators Variance Profile')
plt.show()

In [ ]:
# Statistical Analysis across unique structural crossover pairs
types = list(cx_operators.keys())
for i in range(len(types)):
    for j in range(i+1, len(types)):
        t1, t2 = types[i], types[j]
        d1 = df_crossover.fitness[df_crossover.crossover_type == t1].tolist()
        d2 = df_crossover.fitness[df_crossover.crossover_type == t2].tolist()
        
        _, p1 = shapiro(d1)
        _, p2 = shapiro(d2)
        
        if p1 > 0.05 and p2 > 0.05:
            _, p_val = ttest_ind(d1, d2)
            test_name = "T-Test"
        else:
            _, p_val = mannwhitneyu(d1, d2)
            test_name = "Mann-Whitney"
            
        print(f"{t1} vs {t2} ({test_name}) -> p-value: {p_val:.5f} {'**Significant**' if p_val < 0.05 else 'Not Significant'}")

# Task 3: Exploring effect of tournament size

Conduct an experiment to determine the influence of tournament size using a fixed population size of 100.

In [ ]:
tournament_results = []
tourn_sizes = [2, 4, 8, 16]
toolbox.register("mate", tools.cxOnePoint)

for t_size in tourn_sizes:
    toolbox.register("select", tools.selTournament, tournsize=t_size)
    for reps in range(10):
        pop, log, hof = runEA(custom_popsize=100)
        best = hof[0].fitness.values[0]
        tournament_results.append({'tournament_size': t_size, 'fitness': best})

df_tournament = pd.DataFrame(tournament_results)
df_tournament.boxplot(column=['fitness'], by=['tournament_size'])
plt.title('Selection Pressure: Tournament Size Analysis')
plt.show()

# Task 4: Explore the effect of selection operator:

Comparing Tournament Selection (Size 2) directly against Roulette Wheel Selection (selRoulette).

In [ ]:
selection_results = []
sel_operators = {
    'Tournament_T2': lambda pop, k: tools.selTournament(pop, k, tournsize=2),
    'Roulette_Wheel': tools.selRoulette
}

for name, op in sel_operators.items():
    toolbox.register("select", op)
    for reps in range(10):
        pop, log, hof = runEA(custom_popsize=100)
        best = hof[0].fitness.values[0]
        selection_results.append({'selection_operator': name, 'fitness': best})

df_selection = pd.DataFrame(selection_results)
df_selection.boxplot(column=['fitness'], by=['selection_operator'])
plt.title('Selection Strategy Benchmarks')
plt.show()

In [ ]:
d_t2 = df_selection.fitness[df_selection.selection_operator == 'Tournament_T2'].tolist()
d_roulette = df_selection.fitness[df_selection.selection_operator == 'Roulette_Wheel'].tolist()

_, p_t2 = shapiro(d_t2)
_, p_rou = shapiro(d_roulette)

if p_t2 > 0.05 and p_rou > 0.05:
    _, p_sel = ttest_ind(d_t2, d_roulette)
    test_used = "T-Test"
else:
    _, p_sel = mannwhitneyu(d_t2, d_roulette)
    test_used = "Mann-Whitney"

print(f"Selection Paradigm Comparison via {test_used} -> p-value: {p_sel:.6f}")
if p_sel < 0.05:
    print("Conclusion: Reject Null Hypothesis. Selection dynamics show statistically significant variance.")
else:
    print("Conclusion: Cannot Reject Null Hypothesis. No statistically significant baseline performance gap found.")